# 1b단계: 탐지 실험 — 해상도 · SAHI · YOLO26

0단계 정답(`gt_roboflow.zip`)과 1a ROI를 그대로 씁니다. 실행 순서:

1. 공통 설정 · 평가 준비 (0단계와 같은 셀)
2. 1b-0 → 1b-1 → 1b-2 → 1b-3 (위에서부터 순서대로)
3. 비교표 · 그래프 · 비교 이미지

**런타임: T4 GPU** (전체 약 10~20분)

## 공통 설정

In [ ]:
import os, json, glob, shutil, zipfile, re
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ===== 경로 (v3 노트북 Cell 2와 동일하게) =====
FOLDER_PATH      = "/content/drive/MyDrive/2025년 홍익 vs 국민 드론뷰"
DRIVE_OUTPUT_DIR = os.path.join(FOLDER_PATH, "pipeline_outputs")   # v3의 tracking_final.json 위치
EVAL_DIR         = os.path.join(DRIVE_OUTPUT_DIR, "eval")           # 이번에 새로 만드는 평가 폴더
IMG_DIR          = os.path.join(EVAL_DIR, "images")                 # 평가용 프레임 20장 (정답 라벨의 기준 이미지)
PRE_DIR          = os.path.join(EVAL_DIR, "prelabels")              # 사전 라벨
for _d in (EVAL_DIR, IMG_DIR, PRE_DIR):
    os.makedirs(_d, exist_ok=True)

# ===== 클래스 =====
# 정답(GT)에는 0~2만 남아야 합니다. 3(Unknown)은 "라벨러가 확인해야 할 박스" 표시용입니다.
CLASS_NAMES = ["Hongik", "Kookmin", "Referee", "Unknown"]
CLS_ID = {n: i for i, n in enumerate(CLASS_NAMES)}

# ===== 평가 클립 =====
# play_range : 득점 플레이 구간 (v3 frame_id 기준, 1부터 시작, 양 끝 포함)
# scorer     : v3에서 득점 선수 경로를 만들 때 쓴 (track_id, 프레임 구간). 득점 선수 '정답 위치'로 사용.
#              ⚠️ 0156의 #171, #183 구간은 v3 노트북 주석에 '추정, 확인 필요'로 되어 있었습니다.
#                 v3의 프레임 브라우저 셀로 확인 후 틀렸다면 여기 숫자를 고치세요.
EVAL_CLIPS = {
    "DJI_20250913133612_0144_D": {
        "clip_key": 34,
        "play_range": (413, 649),
        "scorer": [(2122, (413, 649))],
    },
    "DJI_20250913135020_0156_D": {
        "clip_key": 44,
        "play_range": (167, 525),
        "scorer": [(184, (167, 214)), (171, (215, 313)), (183, (314, 393)), (208, (410, 525))],
    },
}
FRAMES_PER_CLIP = 10

# ===== 매칭 기준 =====
# 선수가 10~25px로 작아서 IoU 0.5는 박스가 조금만 어긋나도 오답이 됩니다.
# 대신 '박스 중심 거리'로 짝을 짓습니다: 거리 <= max(MIN_PX, RATIO × 정답 박스 긴 변)
MATCH_MIN_PX = 10
MATCH_RATIO  = 0.5
SCORER_MATCH_PX = 15   # 득점 선수 정답 위치(발)와 예측 박스 발 위치가 이 거리(px) 이내면 '찾았다'


def video_path_of(stem):
    for ext in (".MP4", ".mp4", ".MOV", ".mov"):
        p = os.path.join(FOLDER_PATH, stem + ext)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"영상을 찾을 수 없습니다: {stem}")

def tracking_json_of(stem, folder=DRIVE_OUTPUT_DIR):
    return os.path.join(folder, f"{stem}_tracking_final.json")

for stem in EVAL_CLIPS:
    print(stem, "| 영상:", os.path.exists(video_path_of(stem)), "| v3 JSON:", os.path.exists(tracking_json_of(stem)))

In [ ]:
# ===== 공통 함수 =====

def load_tracking(json_path):
    """v3 형식 JSON → {frame_id: [player dict, ...]}, field_polygon"""
    with open(json_path, encoding="utf-8") as f:
        data = json.load(f)
    by_frame = {fr["frame_id"]: fr["players"] for fr in data["frames"]}
    poly = np.array(data.get("field_polygon", []), dtype=np.int32)
    return by_frame, poly


def read_frames_sequential(video_path, frame_ids):
    """
    frame_id(1부터 시작)에 해당하는 프레임을 '처음부터 순서대로 읽어서' 가져옵니다.
    cap.set(POS_FRAMES)로 바로 점프하면 드론 영상(H.264/H.265)에서 몇 프레임씩 어긋날 수 있어서,
    v3가 프레임을 센 방식(처음부터 1, 2, 3...)과 똑같이 맞추기 위해 순차로 읽습니다.
    """
    wanted = set(frame_ids)
    last = max(wanted)
    out = {}
    cap = cv2.VideoCapture(video_path)
    fid = 0
    while fid < last:
        ok, frame = cap.read()
        if not ok:
            break
        fid += 1
        if fid in wanted:
            out[fid] = frame
    cap.release()
    missing = wanted - set(out)
    if missing:
        print(f"⚠️ 읽지 못한 프레임: {sorted(missing)}")
    return out


def pick_frames(play_range, n):
    a, b = play_range
    return sorted(set(np.linspace(a, b, n).round().astype(int).tolist()))


def inside(poly, x, y):
    if poly is None or len(poly) < 3:
        return True
    return cv2.pointPolygonTest(poly.reshape(-1, 1, 2), (float(x), float(y)), False) >= 0


def xyxy_to_yolo_line(cls_id, box, W, H):
    x1, y1, x2, y2 = box
    x1, x2 = max(0, x1), min(W, x2)
    y1, y2 = max(0, y1), min(H, y2)
    cx, cy = (x1 + x2) / 2 / W, (y1 + y2) / 2 / H
    w, h = (x2 - x1) / W, (y2 - y1) / H
    return f"{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"


def yolo_line_to_xyxy(line, W, H):
    c, cx, cy, w, h = line.split()[:5]
    cx, cy, w, h = float(cx) * W, float(cy) * H, float(w) * W, float(h) * H
    return int(c), [cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2]


def center(b):
    return ((b[0] + b[2]) / 2, (b[1] + b[3]) / 2)


def box_iou(a, b):
    """Day 2에서 만든 IoU (xyxy)"""
    iw = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    ih = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def match_by_center(gt_boxes, pred_boxes):
    """
    정답 박스와 예측 박스를 1:1로 짝짓기 (헝가리안 알고리즘, 중심 거리 기준).
    return: [(gi, pi), ...], 짝 없는 정답 인덱스, 짝 없는 예측 인덱스
    """
    G, P = len(gt_boxes), len(pred_boxes)
    if G == 0 or P == 0:
        return [], list(range(G)), list(range(P))
    gc = np.array([center(b) for b in gt_boxes])
    pc = np.array([center(b) for b in pred_boxes])
    dist = np.linalg.norm(gc[:, None, :] - pc[None, :, :], axis=2)          # (G, P)
    gsize = np.array([max(b[2] - b[0], b[3] - b[1]) for b in gt_boxes])
    thr = np.maximum(MATCH_MIN_PX, MATCH_RATIO * gsize)[:, None]            # 정답 박스별 허용 거리
    cost = np.where(dist <= thr, dist, 1e6)
    gi, pi = linear_sum_assignment(cost)
    pairs = [(g, p) for g, p in zip(gi, pi) if cost[g, p] < 1e6]
    mg = {g for g, _ in pairs}
    mp = {p for _, p in pairs}
    return pairs, [g for g in range(G) if g not in mg], [p for p in range(P) if p not in mp]


print("✅ 공통 함수 준비 완료")

## 평가 준비 (0단계 파트 B와 같은 코드)

In [ ]:
GT_ZIP = os.path.join(EVAL_DIR, "gt_roboflow.zip")
GT_DIR = os.path.join("/content", "gt_extracted")
shutil.rmtree(GT_DIR, ignore_errors=True)
with zipfile.ZipFile(GT_ZIP) as z:
    z.extractall(GT_DIR)


def original_name(fname):
    """Roboflow가 바꾼 파일명 → 원래 이름. 예: 'DJI_..._f00413_jpg.rf.ab12cd.txt' → 'DJI_..._f00413'"""
    base = os.path.basename(fname)
    base = re.split(r"_(?:jpg|jpeg|png)\.rf\.", base, flags=re.IGNORECASE)[0]
    return os.path.splitext(base)[0]


def read_names_yaml(gt_dir):
    """Roboflow data.yaml의 클래스 순서를 읽음 (Roboflow가 순서를 바꿀 수 있어서 이름으로 다시 매핑)"""
    ymls = glob.glob(os.path.join(gt_dir, "**", "data.yaml"), recursive=True)
    if not ymls:
        return CLASS_NAMES
    txt = open(ymls[0], encoding="utf-8").read()
    m = re.search(r"names:\s*\[(.*?)\]", txt, re.S)
    if m:
        return [s.strip().strip("'\"") for s in m.group(1).split(",") if s.strip()]
    return re.findall(r"^\s*-\s*['\"]?([^'\"\n]+?)['\"]?\s*$", txt.split("names:")[1], re.M)


def load_gt(gt_dir):
    """→ {name: {"W","H","boxes":[(class_name, xyxy), ...]}}"""
    rf_names = read_names_yaml(gt_dir)
    gt = {}
    for lbl in glob.glob(os.path.join(gt_dir, "**", "labels", "*.txt"), recursive=True):
        name = original_name(lbl)
        img = cv2.imread(os.path.join(IMG_DIR, name + ".jpg"))   # 원본 해상도 기준으로 복원
        if img is None:
            print("⚠️ 원본 이미지를 찾을 수 없음:", name); continue
        H, W = img.shape[:2]
        boxes = []
        for line in open(lbl).read().strip().splitlines():
            if not line.strip():
                continue
            c, b = yolo_line_to_xyxy(line, W, H)
            boxes.append((rf_names[c], b))
        gt[name] = {"W": W, "H": H, "boxes": boxes}
    return gt


GT = load_gt(GT_DIR)
_cnt = Counter(c for v in GT.values() for c, _ in v["boxes"])
print(f"정답 이미지 {len(GT)}장, 박스 {sum(_cnt.values())}개 → {dict(_cnt)}")
if _cnt.get("Unknown"):
    print(f"⚠️ 정답에 Unknown이 {_cnt['Unknown']}개 남아 있습니다. 탐지 평가에는 포함되지만 팀 정확도에서는 제외됩니다.")

In [ ]:
EVAL_TEAM = ["Hongik", "Kookmin", "Referee"]


def detection_and_team_metrics(run_json_by_stem):
    """정답 vs 예측: 탐지(클래스 무관) P/R/F1 + 짝지어진 박스의 클래스 정확도 + 혼동행렬"""
    tp = fp = fn = 0
    conf_mat = Counter()                     # (정답 클래스, 예측 클래스)
    per_frame = []
    for stem, jpath in run_json_by_stem.items():
        by_frame, _ = load_tracking(jpath)
        for name, g in GT.items():
            if not name.startswith(stem):
                continue
            fid = int(name.split("_f")[-1])
            preds = by_frame.get(fid, [])
            gb = [b for _, b in g["boxes"]]
            pb = [p["bbox"] for p in preds]
            pairs, ug, up = match_by_center(gb, pb)
            tp += len(pairs); fn += len(ug); fp += len(up)
            for gi, pi in pairs:
                conf_mat[(g["boxes"][gi][0], preds[pi]["type"])] += 1
            per_frame.append({"image": name, "gt": len(gb), "pred": len(pb), "TP": len(pairs), "FN": len(ug), "FP": len(up)})
    P = tp / (tp + fp) if tp + fp else 0.0
    R = tp / (tp + fn) if tp + fn else 0.0
    F1 = 2 * P * R / (P + R) if P + R else 0.0
    team_pairs = {k: v for k, v in conf_mat.items() if k[0] in EVAL_TEAM}
    correct = sum(v for (g, p), v in team_pairs.items() if g == p)
    team_acc = correct / sum(team_pairs.values()) if team_pairs else 0.0
    ref_gt = sum(1 for v in GT.values() for c, _ in v["boxes"] if c == "Referee")
    ref_hit = conf_mat.get(("Referee", "Referee"), 0)
    return {
        "det_precision": round(P, 3), "det_recall": round(R, 3), "det_f1": round(F1, 3),
        "TP": tp, "FP": fp, "FN": fn,
        "team_acc": round(team_acc, 3),
        "referee_recall": round(ref_hit / ref_gt, 3) if ref_gt else None,
    }, conf_mat, pd.DataFrame(per_frame)


def scorer_track_metrics(stem, run_json, cfg, verbose=True):
    """
    득점 선수 추적 품질 (라벨링 없이 측정).
    - 정답 위치: v3에서 사람이 확인해 이어붙인 득점 선수의 발 위치 (cfg["scorer"])
    - 평가할 결과(run_json)에서 매 프레임 그 위치에 가장 가까운 박스의 track_id를 읽음

    지표
    - scorer_ids / scorer_id_switches : '정답 위치가 확실한 프레임'에서만 계산 (믿을 수 있는 값)
    - scorer_coverage : v3가 놓친 구간은 앞뒤 위치를 직선으로 이어 정답 위치를 '추정'해서 포함
                        → 그 구간에서는 근처의 다른 선수가 잡힐 수 있으니 참고용(근사값)
    """
    ref_by_frame, _ = load_tracking(tracking_json_of(stem))       # 정답 경로는 항상 v3 기준
    run_by_frame, _ = load_tracking(run_json)
    known = {}
    for tid, (a, b) in cfg["scorer"]:
        for fid in range(a, b + 1):
            for p in ref_by_frame.get(fid, []):
                if p["track_id"] == tid:
                    known[fid] = p["feet"]
    if not known:
        print(f"⚠️ {stem}: v3 JSON에서 득점 선수 track_id를 찾지 못했습니다. EVAL_CLIPS의 scorer 값을 확인하세요.")
        return {"scorer_frames": 0, "scorer_coverage": 0.0, "scorer_ids": 0, "scorer_id_switches": 0}
    span = range(min(known), max(known) + 1)
    kf = np.array(sorted(known))
    kx = np.array([known[f][0] for f in kf]); ky = np.array([known[f][1] for f in kf])

    seq = []   # (frame, 찾은 track_id 또는 None, 정답 위치가 확실한지)
    for fid in span:
        gx, gy = (known[fid] if fid in known else (np.interp(fid, kf, kx), np.interp(fid, kf, ky)))
        best, bd = None, 1e9
        for p in run_by_frame.get(fid, []):
            d = np.hypot(p["feet"][0] - gx, p["feet"][1] - gy)
            if d < bd:
                best, bd = p, d
        tid = best["track_id"] if best is not None and bd <= SCORER_MATCH_PX else None
        seq.append((fid, tid, fid in known))

    sure = [t for _, t, k in seq if k and t is not None]
    switches = sum(1 for a, b in zip(sure, sure[1:]) if a != b)
    covered = sum(1 for _, t, _ in seq if t is not None)

    if verbose:   # 어느 ID가 어느 구간을 맡았는지 (v3 노트북의 SEGMENTS와 같은 형태) → 영상으로 확인할 때 사용
        runs, cur = [], None
        for fid, t, _ in seq:
            if cur and cur[0] == t and cur[2] == fid - 1:
                cur[2] = fid
            else:
                cur = [t, fid, fid]; runs.append(cur)
        desc = ", ".join(f"{'없음' if t is None else '#' + str(t)}({a}~{b})" for t, a, b in runs)
        print(f"  [{stem}] 득점 선수 경로: {desc}")

    return {
        "scorer_frames": len(seq),
        "scorer_coverage": round(covered / len(seq), 3),
        "scorer_ids": len(set(sure)),
        "scorer_id_switches": switches,
    }


def segment_stats(stem, run_json, cfg):
    """플레이 구간 안의 고유 ID 수, Unknown 비율 (라벨링 없이 측정)"""
    by_frame, _ = load_tracking(run_json)
    a, b = cfg["play_range"]
    ids, types = set(), Counter()
    for fid in range(a, b + 1):
        for p in by_frame.get(fid, []):
            ids.add(p["track_id"]); types[p["type"]] += 1
    total = sum(types.values())
    return {"segment_unique_ids": len(ids),
            "segment_unknown_ratio": round(types["Unknown"] / total, 3) if total else 0.0}


RESULTS_CSV = os.path.join(EVAL_DIR, "eval_results.csv")


def evaluate_run(run_name, run_json_by_stem, note=""):
    """
    한 번의 실험 결과를 채점해서 eval_results.csv에 한 줄 추가합니다.
    run_json_by_stem: {stem: 그 실험이 만든 tracking_final.json 경로}
    """
    det, conf_mat, per_frame = detection_and_team_metrics(run_json_by_stem)
    row = {"run": run_name, "note": note, **det}
    for stem, jpath in run_json_by_stem.items():
        cfg = EVAL_CLIPS[stem]
        k = cfg["clip_key"]
        for key, val in {**scorer_track_metrics(stem, jpath, cfg), **segment_stats(stem, jpath, cfg)}.items():
            row[f"{key}_{k}"] = val
    df_new = pd.DataFrame([row])
    if os.path.exists(RESULTS_CSV):
        old = pd.read_csv(RESULTS_CSV)
        old = old[old["run"] != run_name]                      # 같은 이름으로 다시 돌리면 덮어쓰기
        df_new = pd.concat([old, df_new], ignore_index=True)
    df_new.to_csv(RESULTS_CSV, index=False, encoding="utf-8-sig")
    return row, conf_mat, per_frame


def show_confusion(conf_mat):
    rows = EVAL_TEAM + (["Unknown"] if any(g == "Unknown" for g, _ in conf_mat) else [])
    cols = CLASS_NAMES
    return pd.DataFrame([[conf_mat.get((r, c), 0) for c in cols] for r in rows],
                        index=[f"정답:{r}" for r in rows], columns=[f"예측:{c}" for c in cols])

print("✅ 평가 함수 준비 완료")

---
# 1b. 탐지 실험 (평가 프레임 20장에만 탐지)

추적·분류 없이 **탐지기만** 돌려서 정답과 비교합니다. 설정 하나당 몇 분이면 끝나요.

| 실험 | 설정 | 확인할 것 |
|---|---|---|
| 1b-0 | yolov8x, imgsz 960 (v3와 같음) | 놓친 선수 중 **탐지기가 놓친 것** vs **추적기가 버린 것** |
| 1b-1 | yolov8x, imgsz 1280 / 1920 | 해상도를 올리면 작은 선수를 더 찾는지 |
| 1b-2 | yolov8x + SAHI (640 타일) | 잘라서 보면 더 찾는지 |
| 1b-3 | yolo26x, imgsz 960 / 1920 | 최신 모델의 효과 |

- 모든 실험은 **1a에서 만든 ROI**로 걸러서 채점합니다. (1a와 같은 조건)
- 비교는 **conf 0.15**(v3와 같은 값)에서 합니다. 그래프로 다른 conf 값에서의 변화도 같이 봐요.
- 결과는 `det_results.csv`에 따로 쌓입니다. (`eval_results.csv`는 추적까지 한 결과용)

**런타임: T4 GPU**

In [ ]:
!pip install -q ultralytics sahi
import torch
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
DEVICE = 0 if torch.cuda.is_available() else "cpu"

In [ ]:
# 1a에서 확정한 ROI 불러오기 (1a 결과 JSON의 field_polygon)
ROI_1A = {}
for stem in EVAL_CLIPS:
    p = os.path.join(EVAL_DIR, "runs", "step1a_roi", f"{stem}_tracking_final.json")
    assert os.path.exists(p), f"1a 결과가 없습니다: {p} — 1a 노트북을 먼저 실행하세요."
    _, ROI_1A[stem] = load_tracking(p)
    print(stem, "ROI 꼭짓점", len(ROI_1A[stem]))

EVAL_NAMES = sorted(GT)                       # 정답이 있는 20장
CONF_MAIN = 0.15                              # 비교 기준 conf (v3와 같음)
CONF_FLOOR = 0.01                             # 낮게 뽑아두고 나중에 기준별로 거름
DET_CSV = os.path.join(EVAL_DIR, "det_results.csv")
PREDS = {}                                    # 실험 이름 → {이미지 이름: (boxes(N,4), scores(N,))}


def stem_of(name):
    return name.split("_f")[0]


def roi_filter(name, boxes, scores):
    poly = ROI_1A[stem_of(name)]
    keep = [i for i, b in enumerate(boxes) if inside(poly, (b[0] + b[2]) / 2, b[3])]
    return boxes[keep], scores[keep]


def det_metrics(preds, conf):
    """conf 이상 박스만 남겨서 정답과 짝짓기 → TP/FP/FN, 클래스별·크기별 Recall"""
    tp = fp = fn = 0
    cls_tot, cls_hit = Counter(), Counter()
    SIZE_BINS = [(0, 20), (20, 30), (30, 45), (45, 9999)]
    size_tot, size_hit = Counter(), Counter()
    per_img = {}
    for name in EVAL_NAMES:
        boxes, scores = preds.get(name, (np.zeros((0, 4)), np.zeros(0)))
        pb = boxes[scores >= conf].tolist()
        g = GT[name]["boxes"]
        pairs, ug, up = match_by_center([b for _, b in g], pb)
        tp += len(pairs); fn += len(ug); fp += len(up)
        hit = {gi for gi, _ in pairs}
        for gi, (c, b) in enumerate(g):
            h = b[3] - b[1]
            sb = next(f"{lo}~{hi if hi < 9999 else ''}px" for lo, hi in SIZE_BINS if lo <= h < hi)
            cls_tot[c] += 1; size_tot[sb] += 1
            if gi in hit:
                cls_hit[c] += 1; size_hit[sb] += 1
        per_img[name] = (len(pairs), len(ug), len(up))
    P = tp / (tp + fp) if tp + fp else 0.0
    R = tp / (tp + fn) if tp + fn else 0.0
    F1 = 2 * P * R / (P + R) if P + R else 0.0
    out = {"precision": round(P, 3), "recall": round(R, 3), "f1": round(F1, 3), "TP": tp, "FP": fp, "FN": fn}
    for c in ["Hongik", "Kookmin", "Referee"]:
        out[f"recall_{c}"] = round(cls_hit[c] / cls_tot[c], 3) if cls_tot[c] else None
    for sb in sorted(size_tot, key=lambda s: int(s.split("~")[0])):
        out[f"recall_h{sb}"] = round(size_hit[sb] / size_tot[sb], 3)
    return out, per_img


def record(exp_name, note, preds, sec=None):
    PREDS[exp_name] = preds
    m, _ = det_metrics(preds, CONF_MAIN)
    row = {"exp": exp_name, "note": note, "conf": CONF_MAIN, **m, "sec_per_img": sec}
    df = pd.read_csv(DET_CSV) if os.path.exists(DET_CSV) else pd.DataFrame()
    if len(df):
        df = df[df["exp"] != exp_name]
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df.to_csv(DET_CSV, index=False, encoding="utf-8-sig")
    print(f"[{exp_name}] P={m['precision']} R={m['recall']} F1={m['f1']} (TP {m['TP']}, FP {m['FP']}, FN {m['FN']})"
          + (f" | {sec:.2f}s/장" if sec else ""))
    return row


# 비교 기준: v3가 추적까지 해서 남긴 박스 (1a 결과) — 점수가 없으니 모두 1.0으로 취급
ref = {}
for name in EVAL_NAMES:
    by_frame, _ = load_tracking(os.path.join(EVAL_DIR, "runs", "step1a_roi", f"{stem_of(name)}_tracking_final.json"))
    ps = by_frame.get(int(name.split("_f")[-1]), [])
    ref[name] = (np.array([p["bbox"] for p in ps], float).reshape(-1, 4), np.ones(len(ps)))
record("ref_v3_tracked", "v3 최종 결과(탐지+추적) + 1a ROI — 비교 기준", ref)

In [ ]:
import time
from ultralytics import YOLO


def run_yolo(weights, imgsz):
    model = YOLO(weights)
    preds, t0 = {}, time.time()
    for name in EVAL_NAMES:
        r = model.predict(os.path.join(IMG_DIR, name + ".jpg"), imgsz=imgsz, conf=CONF_FLOOR,
                          classes=[0], max_det=300, device=DEVICE, verbose=False)[0]
        b = r.boxes.xyxy.cpu().numpy().reshape(-1, 4); s = r.boxes.conf.cpu().numpy().reshape(-1)
        preds[name] = roi_filter(name, b, s)
    sec = (time.time() - t0) / len(EVAL_NAMES)
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return preds, sec


# ===== 1b-0: v3와 같은 탐지기 설정, 추적 없이 =====
p, sec = run_yolo("yolov8x.pt", 960)
record("1b0_v8x_960", "yolov8x imgsz960 탐지만 (v3 탐지기와 같은 설정)", p, sec)

r_ref = det_metrics(PREDS["ref_v3_tracked"], CONF_MAIN)[0]["recall"]
r_det = det_metrics(PREDS["1b0_v8x_960"], CONF_MAIN)[0]["recall"]
print(f"\n추적까지 한 v3 Recall {r_ref} → 탐지만 Recall {r_det}")
if r_det - r_ref >= 0.05:
    print("→ 탐지기는 찾았는데 추적기가 버린 선수가 꽤 있어요. 2단계(추적기 설정)에서 되찾을 여지가 큽니다.")
else:
    print("→ 놓친 선수 대부분을 탐지기 단계에서부터 못 찾았어요. 탐지기 개선(1b-1~4)이 핵심입니다.")

In [ ]:
# ===== 1b-1: 해상도 올리기 =====
for imgsz in (1280, 1920):
    p, sec = run_yolo("yolov8x.pt", imgsz)
    record(f"1b1_v8x_{imgsz}", f"yolov8x imgsz{imgsz} 탐지만", p, sec)

In [ ]:
# ===== 1b-2: SAHI 타일 추론 =====
# 이미지를 겹치게 잘라(640px, 20% 겹침) 각각 탐지한 뒤 합침 + 원본 전체 탐지도 같이 합침
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

SAHI_SLICE = 640
SAHI_OVERLAP = 0.2


def load_sahi(weights, image_size):
    last = None
    for mtype in ("ultralytics", "yolov8"):       # sahi 버전에 따라 이름이 다름
        try:
            return AutoDetectionModel.from_pretrained(model_type=mtype, model_path=weights,
                                                      confidence_threshold=CONF_FLOOR, image_size=image_size,
                                                      device="cuda:0" if torch.cuda.is_available() else "cpu")
        except Exception as e:
            last = e
    raise last


def run_sahi(weights, slice_size=SAHI_SLICE, overlap=SAHI_OVERLAP):
    dm = load_sahi(weights, slice_size)
    preds, t0 = {}, time.time()
    for name in EVAL_NAMES:
        res = get_sliced_prediction(os.path.join(IMG_DIR, name + ".jpg"), dm,
                                    slice_height=slice_size, slice_width=slice_size,
                                    overlap_height_ratio=overlap, overlap_width_ratio=overlap,
                                    perform_standard_pred=True, postprocess_type="NMS",
                                    postprocess_match_metric="IOS", postprocess_match_threshold=0.5,
                                    verbose=0)
        b, s = [], []
        for op in res.object_prediction_list:
            if op.category.name != "person" and op.category.id != 0:
                continue
            b.append(op.bbox.to_xyxy()); s.append(op.score.value)
        preds[name] = roi_filter(name, np.array(b, float).reshape(-1, 4), np.array(s, float))
    sec = (time.time() - t0) / len(EVAL_NAMES)
    del dm
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return preds, sec


p, sec = run_sahi("yolov8x.pt")
record("1b2_v8x_sahi640", f"yolov8x + SAHI {SAHI_SLICE}px 타일, 겹침 {SAHI_OVERLAP}", p, sec)

In [ ]:
# ===== 1b-3: YOLO26 =====
for imgsz in (960, 1920):
    p, sec = run_yolo("yolo26x.pt", imgsz)
    record(f"1b3_y26x_{imgsz}", f"yolo26x imgsz{imgsz} 탐지만", p, sec)

In [ ]:
# ===== 결과 비교표 =====
det_df = pd.read_csv(DET_CSV)
show_cols = ["exp", "precision", "recall", "f1", "TP", "FP", "FN",
             "recall_Hongik", "recall_Kookmin", "recall_Referee"] + \
            [c for c in det_df.columns if c.startswith("recall_h")] + ["sec_per_img"]
display(det_df[[c for c in show_cols if c in det_df.columns]].set_index("exp"))

In [ ]:
# conf 기준을 바꿨을 때 Precision / Recall 변화 (그래프)
CONF_GRID = np.round(np.arange(0.05, 0.71, 0.05), 2)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for exp, preds in PREDS.items():
    if exp == "ref_v3_tracked":
        continue
    ms = [det_metrics(preds, c)[0] for c in CONF_GRID]
    axes[0].plot([m["recall"] for m in ms], [m["precision"] for m in ms], marker="o", ms=3, label=exp)
    axes[1].plot(CONF_GRID, [m["f1"] for m in ms], marker="o", ms=3, label=exp)
ref_m = det_metrics(PREDS["ref_v3_tracked"], CONF_MAIN)[0]
axes[0].scatter([ref_m["recall"]], [ref_m["precision"]], c="k", marker="*", s=200, label="v3 (탐지+추적)", zorder=5)
axes[0].set_xlabel("Recall"); axes[0].set_ylabel("Precision"); axes[0].set_title("Precision-Recall (conf 0.05~0.7)")
axes[1].axvline(CONF_MAIN, color="gray", ls="--"); axes[1].set_xlabel("conf"); axes[1].set_ylabel("F1"); axes[1].set_title("conf별 F1")
for ax in axes:
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
print("※ 오른쪽 위로 갈수록 좋은 모델. 비교표는 conf 0.15 기준이고, 그래프는 참고용이에요.")
print("  (평가셋 그래프를 보고 conf를 골라 점수를 올리면 시험지에 맞춘 것이 되니, 최종 conf는 다른 프레임으로 정해요.)")

In [ ]:
# 같은 프레임에서 비교: 1b-0(v3 탐지기) vs Recall이 가장 높은 실험
best = det_df[det_df["exp"] != "ref_v3_tracked"].sort_values(["recall", "precision"], ascending=False)["exp"].iloc[0]
print("Recall이 가장 높은 실험:", best)
_, pi0 = det_metrics(PREDS["1b0_v8x_960"], CONF_MAIN)
worst = sorted(EVAL_NAMES, key=lambda n: -pi0[n][1])[:2]          # 1b-0이 가장 많이 놓친 2장


def draw_det(name, exp):
    boxes, scores = PREDS[exp][name]
    pb = boxes[scores >= CONF_MAIN].tolist()
    g = GT[name]["boxes"]
    pairs, ug, up = match_by_center([b for _, b in g], pb)
    img = cv2.imread(os.path.join(IMG_DIR, name + ".jpg"))
    cv2.polylines(img, [ROI_1A[stem_of(name)].reshape(-1, 1, 2)], True, (0, 255, 0), 1)
    for _, pi in pairs:
        b = pb[pi]; cv2.rectangle(img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (0, 200, 0), 2)
    for gi in ug:
        b = g[gi][1]; cv2.rectangle(img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (0, 0, 255), 2)
    for pi in up:
        b = pb[pi]; cv2.rectangle(img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (255, 0, 0), 2)
    return img, len(pairs), len(ug), len(up)


for name in worst:
    fig, axes = plt.subplots(1, 2, figsize=(20, 6))
    for ax, exp in zip(axes, ["1b0_v8x_960", best]):
        img, t, f_n, f_p = draw_det(name, exp)
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax.axis("off")
        ax.set_title(f"{exp}  TP={t} FN={f_n} FP={f_p}", fontsize=11)
    fig.suptitle(f"{name}   초록=찾음  빨강=놓침  파랑=잘못 잡음"); plt.tight_layout(); plt.show()

## 결과 공유
비교표, 그래프, 비교 이미지 2장을 캡처해서 보여주세요. `det_results.csv`를 첨부해도 됩니다.

**해석 가이드**
- `1b0` Recall이 v3(추적까지)보다 많이 높으면 → 추적기가 선수를 버리고 있는 것 (2단계에서 해결)
- `recall_Kookmin`이 다른 팀보다 낮으면 → 진한 유니폼·웅크린 자세 문제 (파인튜닝 1b-4가 필요)
- `recall_h0~20px`처럼 작은 박스의 Recall이 imgsz·SAHI로 오르면 → 해상도 문제였던 것
- `sec_per_img`는 실제 파이프라인 적용 시 처리 시간을 가늠하는 용도예요 (클립 1,600프레임 × 이 값)